In [1]:
pip install nibabel openpyxl

Looking in indexes: http://cor-notebook-dev-wheel-cache.projects:8081/simple, https://download.pytorch.org/whl/cu130
Looking in links: /var/cache/pip/wheels-local, /var/cache/pip/wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import math
import random
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import openpyxl
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
from IPython.display import clear_output

In [3]:
PROJECT_DIR = Path("./")
sys.path.insert(0, str(PROJECT_DIR.resolve()))

TRAIN_DIR = PROJECT_DIR / "Dataset_train_no_process"
VAL_DIR = PROJECT_DIR / "Dataset_validation_no_process"
TRAIN_XLSX = PROJECT_DIR / "train.xlsx"
VAL_XLSX = PROJECT_DIR / "validation.xlsx"
OUTPUT_DIR = PROJECT_DIR / "history_3d_simple_cnn_no_process_4"


In [4]:
label_columns = [
        "ICH" ]

target_size = 128

In [5]:
def show_volume_slice(axarr_, vol_slice, ax_name: str, v_min_max: tuple = (0.0, 1.0)):
    axarr_[0].set_title(f"axis: {ax_name}")
    axarr_[0].imshow(vol_slice, cmap="gray", vmin=v_min_max[0], vmax=v_min_max[1])
    axarr_[1].plot(torch.sum(vol_slice, 1), list(range(vol_slice.shape[0]))[::-1])
    axarr_[1].plot(list(range(vol_slice.shape[1])), torch.sum(vol_slice, 0))
    axarr_[1].set_aspect("equal")
    axarr_[1].grid()


def idx_middle_if_none(volume, *xyz):
    xyz = list(xyz)
    vol_shape = volume.shape
    for i, d in enumerate(xyz):
        if d is None:
            xyz[i] = int(vol_shape[i] / 2)
        assert 0 <= xyz[i] < vol_shape[i]
    return xyz


def show_volume(
    volume,
    x = None,
    y = None,
    z = None,
    fig_size = (14, 9),
    v_min_max = (0.0, 1.0),
):
    """Show volume in the three axis/cuts.
    """
    x, y, z = idx_middle_if_none(volume, x, y, z)
    fig, axarr = plt.subplots(nrows=2, ncols=3, figsize=fig_size)
    print(f"shape: {volume.shape}, x={x}, y={y}, z={y}  >> {volume.dtype}")
    show_volume_slice(axarr[:, 0], volume[x, :, :], "X", v_min_max)
    show_volume_slice(axarr[:, 1], volume[:, y, :], "Y", v_min_max)
    show_volume_slice(axarr[:, 2], volume[:, :, z], "Z", v_min_max)
    # plt.show(fig)
    return fig

In [6]:
xl_file = pd.ExcelFile(TRAIN_XLSX)

samples_df = {sheet_name: xl_file.parse(sheet_name) 
        for sheet_name in xl_file.sheet_names}['Sheet1']

for index, row in samples_df.iterrows():
    study_uid = str(row.study_uid)
    image_path = TRAIN_DIR / f"{study_uid}.nii.gz"
    if image_path.exists():
        samples_df.at[index, "image_path"] = str(image_path)

samples_df = samples_df.dropna(subset=["image_path"])

In [7]:
# for index, row in samples_df[:1].iterrows():
#     volume = nib.load(row["image_path"]).get_fdata(dtype="float32")
#     volume = torch.from_numpy(volume)

    # Crop and resize volume
    # volume = crop_volume(volume, thr=1e-6)
    # volume = resize_volume(volume, size=target_size)

    # print(volume.shape, volume.mean(), volume.td())
    

In [8]:
# fig = show_volume(volume, fig_size=(9, 6))

In [9]:
from dataset_no_process import CTVolumeDataset

train_dataset = CTVolumeDataset(
    xlsx_path=TRAIN_XLSX,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    xlsx_path=VAL_XLSX,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

# print(f"Train samples: {len(train_dataset)}")
# x, y = train_dataset[0]
# print(f"Volume shape: {x.shape}, dtype: {x.dtype}")
# print(f"Labels shape: {y.shape}, labels: {y.tolist()}")


### Проверка даталодер

In [10]:
# torch.multiprocessing.set_sharing_strategy('file_system')

# # Training configuration
# batch_size = 8
# num_epochs = 50
# print_every_i = 10  
# learning_rate = 1e-4
# num_workers = 4
# loss_log_every_i = 10

# # Device setup
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # DataLoader
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=batch_size,
#     shuffle=True,
#     num_workers=num_workers,
#     prefetch_factor=1,
#     pin_memory=False,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=batch_size,
#     shuffle=False,
#     num_workers=num_workers,
#     prefetch_factor=2,
#     pin_memory=False,
# )


# for i, (volumes, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {1}/{num_epochs}", leave=False), start=1):
#     volumes = volumes.to(device)
#     labels = labels.to(device)
#     print(volumes.shape, labels.shape)

#     if i == 5:
#         break

In [11]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class Simple3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()

        self.encoder = nn.Sequential(
            # 128 → 64
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),

            # 64 → 32
            ConvBlock3D(16, 32, stride=2,  dropout_p=0.1),

            # 32 → 16
            ConvBlock3D(32, 64, stride=2,  dropout_p=0.1),

            # 16 → 8
            ConvBlock3D(64, 128, stride=2,  dropout_p=0.1),

            # 8 → 4
            ConvBlock3D(128, 256, stride=2,  dropout_p=0.1),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),  # → (B, 256, 1, 1, 1)
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x

In [12]:
# Training configuration
batch_size = 8
num_epochs = 50
print_every_i = 10  
learning_rate = 1e-4
num_workers = 4
loss_log_every_i = 10
prefetch_factor = 2

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    prefetch_factor = prefetch_factor,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    prefetch_factor = prefetch_factor,
    pin_memory=(device.type == "cuda"),
)

# Model, loss, optimizer

model = Simple3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)

criterion = nn.BCEWithLogitsLoss()

start_epoch = 1

# checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{start_epoch - 1}.pt", map_location=device)

optimizer = AdamW(model.parameters(), lr=learning_rate)

# model.load_state_dict(checkpoint["model_state_dict"])

# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

epoch_metrics_path = OUTPUT_DIR / "epoch_metrics_no_process.csv"
iter_losses_path = OUTPUT_DIR / "iter_losses_every_10_no_process.csv"


# Training loop
for epoch in range(start_epoch, num_epochs + 1):
    model.train()
    epoch_loss_sum = 0.0

    iter_losses_current_epoch = []

    for i, (volumes, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False), start=1):
        volumes = volumes.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(volumes)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        iter_loss = round(loss.item(), 5)
        epoch_loss_sum += loss.item()

        if i % loss_log_every_i == 0:
            iter_losses_current_epoch.append(
                {
                    "epoch": epoch,
                    "split": "train",
                    "iteration": i,
                    "loss": iter_loss,
                }
            )


    epoch_loss = round(epoch_loss_sum / max(len(train_loader), 1), 5)

    model.eval()
    val_loss_sum = 0.0

    with torch.no_grad():
        for i, (volumes, labels) in enumerate(val_loader, start=1):
            volumes = volumes.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(volumes)
            loss = criterion(logits, labels)
            val_loss_sum += loss.item()

    val_loss = round(val_loss_sum / max(len(val_loader), 1), 5)

    epoch_row = pd.DataFrame(
        [{
            "epoch": epoch,
            "train_loss": epoch_loss,
            "val_loss": val_loss,
        }]
    )

    epoch_row.to_csv(
        epoch_metrics_path,
        mode="a",
        header=not epoch_metrics_path.exists(),
        index=False
    )

    if iter_losses_current_epoch:
        pd.DataFrame(iter_losses_current_epoch).to_csv(
            iter_losses_path,
            mode="a",
            header=not iter_losses_path.exists(),
            index=False
        )

    
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt"
    )



Using device: cuda


KeyboardInterrupt: 